# 05 — Matching: filters -> criteria -> score -> reasons -> notifications

> **Restart kernel + Run All.** After editing any `.py` file: restart the kernel,
> then Run All. This notebook deliberately does not use `%autoreload` — reloading a
> module mid-session leaves two copies of the same Pydantic class alive and breaks
> validation.

> **Cost gate:** the model is only ever asked to REPHRASE reasons (step 5), only
> with `RUN_LIVE_LLM=1`, and only for a capped number of the best matches.
> Filters, criteria, embeddings, scoring and notifications are deterministic and
> local. Step 6 (your own CV) forces the offline provider, so nothing you own
> can leave the machine even when the gate is open.

**What this turns into:** 32 opportunities x 5 profiles -> ranked matches per profile

- **hard filters** first: expired, scam-flagged or against the declared preferences
  are dropped with a human-readable reason. Missing data never filters — an unknown
  country or a null deadline stays, flagged for verification;
- **criteria rules**: every requirement line becomes MET / MISSED / UNKNOWN with
  evidence. Absence of data is UNKNOWN (never a guess), contradicting data MISSED;
- **score 0-100** = `0.5 x semantic similarity + 0.35 x criteria ratio + 0.15 x bonus`
  (bonus = trust / language / funding signals) — every score is recomputable from
  the parts stored next to it;
- **reasons** are written by the rules; the model may only reword them (same facts,
  same count) for matches >= `NOTIFY_THRESHOLD`, behind the cost gate;
- **notifications**: threshold, daily cap, quiet hours 22-08, digest mode and
  "never twice for the same opportunity" — one pure decision function.

> **Privacy:** an opportunity's public title/organizer and the short rule reasons MAY
> be sent to Gemini (step 5, gated). Your CV and profile NEVER are.


In [ ]:
import json
import os
import sys
from datetime import date, datetime
from pathlib import Path

# Make oppradar_ai importable whether the kernel starts in ai/ or ai/notebooks/.
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))

import pandas as pd
from IPython.display import display

from oppradar_ai.config import Settings, get_settings, setup_logging
from oppradar_ai.llm import get_llm
from oppradar_ai.matching import (
    apply_hard_filters,
    check_criteria,
    combine,
    criteria_ratio,
    decide_notifications,
    evaluate_line,
    prefs_from_settings,
    score_user,
)
from oppradar_ai.matching.reasons import phrase_reasons, phrasing_allowed
from oppradar_ai.schemas import (
    MatchResult,
    Notification,
    NotificationMode,
    NotificationStatus,
    Opportunity,
    Profile,
    ScamFlag,
    ScamFlagDetail,
    TrustLevel,
)

# -- Cost gate: only step 5 may call the model, and only when you opt in -------
RUN_LIVE_LLM = Settings().run_live_llm
os.environ.pop('LLM_PROVIDER', None)      # drop the force of a previous run
_configured = Settings().llm_provider     # fresh read of ai/.env (get_settings is cached)
_can_live = _configured != 'gemini' or bool(Settings().gemini_key_value)
os.environ['LLM_PROVIDER'] = _configured if RUN_LIVE_LLM and _can_live else 'fake'
get_settings.cache_clear()

setup_logging('INFO')
settings = get_settings().ensure_dirs()

# Scoring demos (steps 3, 4, 6) must never spend a phrasing call: only step 5
# opts back in. Otherwise the 5-profile sweep would burn the free tier's
# per-minute quota before the phrasing demo gets its turn.
SWEEP = settings.model_copy(update={'run_live_llm': False})

FIXTURES = settings.data_dir / 'fixtures'
opps = [
    Opportunity.model_validate(o)
    for o in json.loads((FIXTURES / 'opportunities.json').read_text(encoding='utf-8'))
]
profs = [
    Profile.model_validate(p)
    for p in json.loads((FIXTURES / 'profiles.json').read_text(encoding='utf-8'))
]
TODAY = date.today()

# The scam detector (notebook 03) writes trust flags into the stored records;
# the fixtures stay neutral on purpose, so we replay its verdicts here, offline.
scam_labels = json.loads((settings.eval_dir / 'scam_labels.json').read_text(encoding='utf-8'))
for opp in opps:
    if scam_labels.get(opp.id) is True:
        opp.trust_level = TrustLevel.SCAM
        opp.scam_flags = [ScamFlagDetail(flag=ScamFlag.UNVERIFIABLE_ORGANIZER,
                                         detail='replayed from data/eval/scam_labels.json')]

amine = next(p for p in profs if p.user_id == 'u-amine')
print(f'fixtures: {len(opps)} opportunities x {len(profs)} profiles | pinned today: {TODAY}')
REAL_LLM = settings.llm_provider == 'gemini' and bool(settings.gemini_key_value)
print('live LLM:', 'yes' if REAL_LLM else 'no',
      '| provider:', settings.llm_provider,
      '| step 5:', 'LIVE' if RUN_LIVE_LLM else 'OFFLINE')


## Step 1 — Hard filters: what cannot fit is removed before any work

`apply_hard_filters(profile, opportunities)` splits the list into (kept,
dropped-with-reason). Expired deadlines and scam-flagged records go; so does an
opportunity that contradicts the preferences the user declared (type, mode,
country, needs funding). **Missing** data never filters: an unknown country, an
unstated mode or a null deadline stays, flagged for verification.


In [ ]:
from collections import Counter

rows = []
for profile in profs:
    outcome = apply_hard_filters(profile, opps, today=TODAY)
    counts = Counter(d.reason.value for d in outcome.dropped)
    rows.append({'profile': profile.user_id, 'kept': len(outcome.passed),
                 'dropped': len(outcome.dropped), **counts})
summary = pd.DataFrame(rows).fillna(0)
for column in summary.columns:
    if column != 'profile':
        summary[column] = summary[column].astype(int)
display(summary)

outcome = apply_hard_filters(amine, opps, today=TODAY)
print(f'u-amine keeps {len(outcome.passed)} of {len(opps)} — every drop explains itself:')
display(pd.DataFrame([
    {'id': d.opportunity_id, 'title': d.title[:36], 'why': d.reason.value, 'detail': d.detail}
    for d in outcome.dropped
]).head(8))


## Step 2 — Criteria: every requirement line, checked by rules

`evaluate_line(profile, line)` runs a chain of small rule functions (no LLM):
languages, degree level and field, years of experience, "no requirement" lines,
age (never stored — flagged for privacy), personal constraints, geography, and
finally a fuzzy keyword match. `check_criteria(profile, opportunity)` applies
them to every eligibility line of one listing.

- **MET / MISSED** are counted toward the score, **UNKNOWN** never is;
- a requirement the profile cannot show is UNKNOWN, not a silent failure.


In [ ]:
demo_lines = [
    'English B2 required', 'Japanese required', 'Python', 'React knowledge',
    'No skills needed', 'Open to all', 'Ages 18-34 only', 'Willing to relocate',
    'Two years of work experience', "Master's students in ML or CS",
    'MSc in CS or NLP', 'African residents', 'Portfolio required',
]
rows = []
for line in demo_lines:
    c = evaluate_line(amine, line)
    rows.append({'requirement': line, 'status': c.status.value.upper(), 'evidence': c.evidence})
display(pd.DataFrame(rows))

top_opp = next(o for o in opps if o.id == 'opp-01')
print(f'\nAll eligibility lines of {top_opp.id} — {top_opp.title}')
display(pd.DataFrame([
    {'requirement': c.text, 'status': c.status.value, 'evidence': c.evidence}
    for c in check_criteria(amine, top_opp)
]))


## Step 3 — The 0-100 score and its reasons

`score_user(profile, opportunities)` = hard filters -> criteria -> embedding
similarity (local sentence-transformers, cosine) -> weighted score. The weights
live in the configuration (`SEMANTIC_WEIGHT` / `CRITERIA_WEIGHT` / `BONUS_WEIGHT`),
so every score below can be recomputed from the parts stored next to it — the
formula, not a black box.

Reasons come from the rules first: met requirements, real skill overlap, and the
bonus signals (funding, language, verified organizer). At most three, the ones
that add the most to the score.


In [ ]:
print(f'score_user: {len(profs)} profiles x {len(opps)} opportunities')
rows = []
# SWEEP closes the phrasing gate: every reason below is the rules' own work.
for profile in profs:
    for match in score_user(profile, opps, settings=SWEEP, today=TODAY)[:3]:
        rows.append({
            'profile': profile.user_id, 'id': match.opportunity.id,
            'title': match.opportunity.title[:32], 'score': match.score,
            'sim': match.semantic_similarity,
            'ratio': criteria_ratio(match.criteria), 'bonus': match.bonus,
            'reason 1': match.top_reasons[0][:46],
        })
display(pd.DataFrame(rows))

# Every score is recomputable from its parts (same rounding, same weights).
checks = [
    match.score == combine(match.semantic_similarity, criteria_ratio(match.criteria),
                           match.bonus, settings)
    for profile in profs
    for match in score_user(profile, opps, settings=SWEEP, today=TODAY)
]
print(f'score reproducible from its parts: {all(checks)} ({sum(checks)}/{len(checks)} matches)')


## Step 4 — Notifications: threshold, cap, quiet hours, digest, dedupe

`decide_notifications(matches, prefs, history, *, now=None)` is a **pure**
decision function: same inputs -> same outputs, nothing sent, nothing stored,
nothing slept on. The rules, in order: below the threshold is not a notification;
already notified for this user + opportunity -> SUPPRESSED; digest mode ->
PENDING with the next digest; quiet hours (22-08) -> PENDING; the daily cap ->
PENDING deferred to tomorrow; otherwise SENT.

`prefs_from_settings(profile_id)` fills the preferences from the configuration —
nothing is hardcoded here.


In [ ]:
NOON = datetime(2026, 10, 21, 12, 0)
prefs = prefs_from_settings('u-notebook-demo')

# A synthetic shortlist makes every rule visible whatever the embeddings scored.
demo_matches = [
    MatchResult(profile_id='u-notebook-demo', opportunity=next(o for o in opps if o.id == oid),
                score=score, semantic_similarity=0.5, top_reasons=[f'Rule reason for {oid}'])
    for oid, score in (('opp-01', 92.0), ('opp-02', 85.0), ('opp-14', 71.0), ('opp-03', 64.0))
]

print(f'1) threshold {prefs.threshold}: the 64.0 match is not a notification at all')
notes = decide_notifications(demo_matches, prefs, [], now=NOON)
display(pd.DataFrame([
    {'match': n.opportunity_id, 'score': n.score, 'status': n.status.value,
     'why': n.suppressed_reason or '-'}
    for n in notes
]))

print('2) quiet hours 22-08 defer the same list instead of waking anyone')
deferred = decide_notifications(demo_matches, prefs, [], now=datetime(2026, 10, 21, 23, 30))
print('  ->', [n.status.value for n in deferred])

print('3) digest mode queues everything for the next digest')
digest = prefs.model_copy(update={'mode': NotificationMode.DIGEST})
print('  ->', [n.status.value for n in decide_notifications(demo_matches, digest, [], now=NOON)])

print("4) the daily cap: cap 2 today, and yesterday's 5 sends do not count")
history = [
    Notification(profile_id='u-notebook-demo', opportunity_id=f'past-{i}', title=f'past {i}',
                 score=80.0, status=NotificationStatus.SENT,
                 created_at=datetime(2026, 10, 20, 10, 0))
    for i in range(1, 6)
]
capped = prefs.model_copy(update={'daily_cap': 2})
notes = decide_notifications(demo_matches, capped, history, now=NOON)
print('  ->', [(n.opportunity_id, n.status.value) for n in notes])

print('5) never twice for the same user + opportunity')
already = [Notification(profile_id='u-notebook-demo', opportunity_id='opp-01', title='sent before',
                        score=92.0, status=NotificationStatus.SENT, created_at=NOON)]
note = decide_notifications(demo_matches[:1], prefs, already, now=NOON)[0]
print('  ->', note.status.value, '|', note.suppressed_reason)

print('6) the real shortlist for u-amine today')
real = decide_notifications(score_user(amine, opps, settings=SWEEP, today=TODAY),
                            prefs_from_settings(amine.user_id), [], now=NOON)
print(f'  -> {len(real)} notifications (threshold {prefs.threshold}, cap {prefs.daily_cap})')


## Step 5 — LLM phrasing (gated): reword, never re-invent

The rules' reasons are complete on their own. With `RUN_LIVE_LLM=1` the model is
asked to reword the reasons of the best matches (score >= `NOTIFY_THRESHOLD`,
at most `MAX_PHRASES_PER_RUN` per run) — same count, same order, same facts, and
it only ever sees the short reason lines plus the listing's public title and
organizer. No CV, no name, no profile. Without the gate the cell simply says so.


In [ ]:
settings = get_settings()
matches = score_user(amine, opps, today=TODAY)   # the gated phrasing runs inside
phrased = [m for m in matches if m.reasons_source == 'llm']
gate = 'OPEN' if phrasing_allowed(settings) else 'closed'
print(f'gate {gate}: rephrased {len(phrased)} of {len(matches)} matches '
      f'(cap {settings.max_phrases_per_run}, threshold {settings.notify_threshold})')

if phrasing_allowed(settings):
    for match in phrased[:2]:
        print(f'--- {match.opportunity.id}: {match.opportunity.title[:48]}')
        for line in match.top_reasons:
            print('   ', line)

    # The same call on purpose-made rule reasons, so both versions stay visible:
    demo_opp = next(o for o in opps if o.id == 'opp-01')
    demo_reasons = ['Requirement met: Python', 'Funding is offered',
                    'Deadline 2026-12-05 (as stated in the source)']
    print('\nstandalone phrase_reasons() — rules in, reworded out:')
    reworded = phrase_reasons(demo_reasons, demo_opp, get_llm())
    if reworded == demo_reasons:
        print('   (the model did not answer this time — the rules reasons are kept)')
    for line in reworded:
        print('   ', line)
else:
    print('phrasing skipped (offline): set RUN_LIVE_LLM=1 to reword reasons with Gemini')


## Step 6 — Score YOUR OWN CV (local only, never committed, never sent)

Put your (anonymized) CV in `ai/data/private/` and change the file name below.
This cell forces `LLM_PROVIDER=fake` around the whole document pipeline — text
layer, local OCR and extraction — so nothing you own can reach a remote model,
**even with `RUN_LIVE_LLM=1`**. The folder is git-ignored; nothing in it is ever
committed.

If the file is missing the cell prints how to add one and the rest of the
notebook is unaffected.


In [ ]:
# 1) Put your file in ai/data/private/ and change the file name below.
MY_FILE = settings.data_dir / 'private' / 'french_resume.pdf'   # .pdf, .docx, .png or .jpg

import contextlib


@contextlib.contextmanager
def _offline_llm():
    """Run one block with the fake provider, whatever ai/.env says."""
    previous = os.environ.get('LLM_PROVIDER')
    os.environ['LLM_PROVIDER'] = 'fake'
    get_settings.cache_clear()
    try:
        yield
    finally:
        if previous is not None:
            os.environ['LLM_PROVIDER'] = previous
        get_settings.cache_clear()


if not MY_FILE.exists():
    print(f'File not found: {MY_FILE}')
    print('Create the folder data/private/ and put your (anonymized) CV in it.')
else:
    from oppradar_ai.documents import build_profile, extract_cv, read_document

    with _offline_llm():                      # <- nothing you own leaves this machine
        document = read_document(MY_FILE)     # text layer -> local OCR -> no vision model
        cv = extract_cv(document.text, document.file_name)
        me = build_profile([cv], user_id='u-me').profile

    print('provider used for your CV: fake (offline, always)')
    print('headline :', me.headline)
    print('skills   :', me.skills)
    print('languages:', me.languages)

    mine = score_user(me, opps, settings=SWEEP, today=TODAY)
    print(f'\n{len(mine)} of {len(opps)} opportunities survive the filters for your profile')
    for match in mine[:5]:
        print(f'  {match.score:5.1f}  {match.opportunity.id}  {match.opportunity.title[:44]}')
        for reason in match.top_reasons:
            print('         -', reason)


## Expected output

- **top of the notebook**: `live LLM: no | provider: fake | step 5: OFFLINE`
  unless `RUN_LIVE_LLM=1` (+ key) is set;
- **fixtures line**: `32 opportunities x 5 profiles`, with today's real date
  (the fixtures' deadlines are relative to 2026-10-21 — expired ones drop as
  time passes, which is the point);
- **step 1**: one row per profile with `kept` / `dropped` and a per-reason
  breakdown (`expired`, `scam`, `type_mismatch`, `mode_mismatch`,
  `country_mismatch`, `no_funding` — columns appear only when non-zero), then
  u-amine's first 8 drops, each with its human-readable detail. The two
  scam-labelled fixtures (`opp-31`, `opp-32`) show `scam` for everyone;
- **step 2**: a 13-row table — `English B2 required` MET, `Japanese required`
  MISSED, `Ages 18-34 only` UNKNOWN (privacy), `Two years of work experience`
  MISSED for u-amine (~0.2 year listed), `Portfolio required` UNKNOWN — then
  every eligibility line of `opp-01` with its evidence;
- **step 3**: a table with the top-3 matches of each profile (score, sim, ratio,
  bonus, first reason) and the line
  `score reproducible from its parts: True (N/N matches)` — scores differ
  between machines only through the embedding model, never through hidden state.
  Every reason here is the **rules' own** work: steps 3, 4 and 6 close the
  phrasing gate, so a live run spends no quota on them (only step 5 does);
- **step 4**: the synthetic shortlist through every rule — 64.0 filtered out by
  the threshold, 23:30 deferred by quiet hours, digest all-PENDING, cap 2
  (yesterday's sends ignored), `opp-01` SUPPRESSED as a duplicate — then the
  real notification count for u-amine today (with the default threshold 70 only
  genuinely strong matches qualify);
- **step 5 offline**: `gate closed: ...` + `phrasing skipped (offline): set
  RUN_LIVE_LLM=1 to reword reasons with Gemini`. Live: up to 5 matches reworded
  (one call each, answered by `GEMINI_MODEL`; a repeat run is free — the answers
  are disk-cached), rules kept everywhere else. If the quota is gone the cell
  says `(the model did not answer this time — the rules reasons are kept)`
  instead of pretending the model answered;
- **step 6**: without a private CV, `File not found: .../data/private/...` and
  how to add one; with your own file, the profile it produced and its top-5
  matches — the provider line always says `fake`, whatever `.env` holds;
- if `BAAI/bge-m3` cannot load (it needs torch >= 2.6), the warning names the
  reason and the configured `EMBEDDING_FALLBACK_MODEL`
  (paraphrase-multilingual-MiniLM-L12-v2) loads automatically — set
  `EMBEDDING_MODEL` to the fallback in `ai/.env` to skip the failed attempt;
- offline (fake / no keys): everything above runs on the committed fixtures,
  with the real **local** embedding model (no network, no download);
- nothing raises — a missing file or an empty shortlist becomes a message.
